In [1]:
import subprocess, sys, os

# 装 APScheduler
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "apscheduler"],
    capture_output=True, text=True
)
print(result.stdout[-300:] if result.stdout else "")
print(result.stderr[-200:] if result.stderr else "")

# 建目录
for d in ["src", "data", "output"]:
    os.makedirs(os.path.expanduser(f"~/ml-projects/stock-ranker/{d}"), exist_ok=True)
    
print("目录结构：")
os.system("ls ~/ml-projects/stock-ranker/")

-none-any.whl (64 kB)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [apscheduler]



目录结构：
data
notebooks
output
src


0

In [2]:
src_path = os.path.expanduser("~/ml-projects/stock-ranker/src/ranker.py")

code = '''
import yfinance as yf
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import pickle
from datetime import datetime

class ListNet(nn.Module):
    def __init__(self, input_dim=8):
        super().__init__()
        self.scorer = nn.Sequential(
            nn.Linear(input_dim, 32), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(32, 16), nn.ReLU(),
            nn.Linear(16, 1)
        )
    def forward(self, x):
        return self.scorer(x).squeeze(1)

class StockRanker:
    FEATURE_COLS = ['mom_5d', 'mom_10d', 'mom_20d', 'vol_10d', 'vol_20d',
                    'rsi_dist', 'high_20d_ratio', 'vol_ratio']
    
    def __init__(self, model_path, scaler_path, tickers):
        self.tickers = tickers
        
        # 加载模型
        ckpt = torch.load(model_path, map_location="cpu")
        self.model = ListNet(input_dim=8)
        self.model.load_state_dict(ckpt["model_state"])
        self.model.eval()
        
        # 加载 scaler
        with open(scaler_path, "rb") as f:
            self.scaler = pickle.load(f)
        
        print(f"[StockRanker] 加载完成，股票池: {len(tickers)} 只")
    
    def _fetch_features(self):
        """拉取最新数据并计算特征，返回最新一天的特征 DataFrame"""
        raw = yf.download(self.tickers, period="2mo", interval="1d",
                          auto_adjust=True, progress=False)
        close  = raw["Close"]
        volume = raw["Volume"]
        
        all_feats = []
        for ticker in self.tickers:
            f = pd.DataFrame(index=close.index)
            f["mom_5d"]         = close[ticker].pct_change(5)
            f["mom_10d"]        = close[ticker].pct_change(10)
            f["mom_20d"]        = close[ticker].pct_change(20)
            f["vol_10d"]        = close[ticker].pct_change().rolling(10).std()
            f["vol_20d"]        = close[ticker].pct_change().rolling(20).std()
            f["rsi_dist"]       = (close[ticker] - close[ticker].rolling(20).mean()) / close[ticker].rolling(20).std()
            f["high_20d_ratio"] = close[ticker] / close[ticker].rolling(20).max()
            f["vol_ratio"]      = volume[ticker] / volume[ticker].rolling(10).mean()
            f["ticker"]         = ticker
            all_feats.append(f)
        
        df = pd.concat(all_feats).dropna()
        # 取最新一天
        latest_date = df.index.max()
        latest = df.loc[latest_date].reset_index(drop=True)
        return latest, latest_date
    
    def run(self, top_k=5):
        """执行一次推理，返回 watch list"""
        latest, latest_date = self._fetch_features()
        
        # 标准化
        feats_scaled = self.scaler.transform(latest[self.FEATURE_COLS].values)
        feats_tensor = torch.tensor(feats_scaled, dtype=torch.float32)
        
        # 打分排序
        with torch.no_grad():
            scores = self.model(feats_tensor).numpy()
        
        latest["score"] = scores
        watch_list = (latest[["ticker", "score"]]
                      .sort_values("score", ascending=False)
                      .head(top_k)
                      .reset_index(drop=True))
        watch_list.index += 1  # rank 从1开始
        
        print(f"\\n[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] Watch List ({latest_date.date()})")
        print(watch_list.to_string())
        return watch_list, latest_date
'''

with open(src_path, "w") as f:
    f.write(code)
print("已保存 src/ranker.py")

已保存 src/ranker.py


In [3]:
import sys
sys.path.insert(0, os.path.expanduser("~/ml-projects/stock-ranker/src"))

from ranker import StockRanker

tickers = [
    "AAPL", "MSFT", "GOOGL", "AMZN", "NVDA",
    "META", "TSLA", "AMD",  "INTC", "CRM",
    "JPM",  "BAC",  "GS",   "MS",   "WFC",
    "JNJ",  "UNH",  "PFE",  "MRK",  "ABBV"
]

ranker = StockRanker(
    model_path  = os.path.expanduser("~/ml-projects/stock-ranker/data/listnet_v2.pth"),
    scaler_path = os.path.expanduser("~/ml-projects/stock-ranker/data/scaler_v2.pkl"),
    tickers     = tickers
)

watch_list, date = ranker.run(top_k=5)

[StockRanker] 加载完成，股票池: 20 只

[2026-04-14 21:32:29] Watch List (2026-04-14)
  ticker     score
1   AMZN  0.207208
2    WFC  0.174811
3   NVDA  0.159055
4     MS  0.146030
5   META  0.107761


/opt/anaconda3/envs/mlenv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [4]:
pipeline_path = os.path.expanduser("~/ml-projects/stock-ranker/src/pipeline.py")

code = '''
import os, sys, json
from datetime import datetime
from apscheduler.schedulers.blocking import BlockingScheduler

sys.path.insert(0, os.path.dirname(__file__))
from ranker import StockRanker

TICKERS = [
    "AAPL", "MSFT", "GOOGL", "AMZN", "NVDA",
    "META", "TSLA", "AMD",  "INTC", "CRM",
    "JPM",  "BAC",  "GS",   "MS",   "WFC",
    "JNJ",  "UNH",  "PFE",  "MRK",  "ABBV"
]

DATA_DIR   = os.path.expanduser("~/ml-projects/stock-ranker/data")
OUTPUT_DIR = os.path.expanduser("~/ml-projects/stock-ranker/output")

ranker = StockRanker(
    model_path  = f"{DATA_DIR}/listnet_v2.pth",
    scaler_path = f"{DATA_DIR}/scaler_v2.pkl",
    tickers     = TICKERS
)

def daily_job():
    watch_list, date = ranker.run(top_k=5)
    
    # 保存到 output/YYYY-MM-DD.json
    out = {
        "date": str(date.date()),
        "generated_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "watch_list": watch_list.rename(columns={"score": "model_score"})
                                .to_dict(orient="records")
    }
    out_path = f"{OUTPUT_DIR}/{date.date()}.json"
    with open(out_path, "w") as f:
        json.dump(out, f, indent=2)
    print(f"[pipeline] 已保存 {out_path}")

if __name__ == "__main__":
    # 立即跑一次
    print("[pipeline] 启动，立即执行一次...")
    daily_job()
    
    # 每个工作日 18:00 触发（美股收盘后）
    scheduler = BlockingScheduler(timezone="America/New_York")
    scheduler.add_job(daily_job, "cron",
                      day_of_week="mon-fri", hour=18, minute=0)
    print("[pipeline] 调度器启动，每个工作日 18:00 ET 执行")
    print("[pipeline] Ctrl+C 停止")
    try:
        scheduler.start()
    except KeyboardInterrupt:
        print("[pipeline] 已停止")
'''

with open(pipeline_path, "w") as f:
    f.write(code)
print("已保存 src/pipeline.py")

已保存 src/pipeline.py


In [6]:
from datetime import datetime
import json

output_dir = os.path.expanduser("~/ml-projects/stock-ranker/output")

# 模拟 daily_job
watch_list, date = ranker.run(top_k=5)

out = {
    "date": str(date.date()),
    "generated_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "watch_list": watch_list.rename(columns={"score": "model_score"})
                            .to_dict(orient="records")
}

out_path = f"{output_dir}/{date.date()}.json"
with open(out_path, "w") as f:
    json.dump(out, f, indent=2)

print(f"已保存 {out_path}")
print("\n内容：")
print(json.dumps(out, indent=2))


[2026-04-14 21:35:36] Watch List (2026-04-14)
  ticker     score
1   AMZN  0.207208
2    WFC  0.174811
3   NVDA  0.159055
4     MS  0.146030
5   META  0.107761
已保存 /Users/tongwenchao/ml-projects/stock-ranker/output/2026-04-14.json

内容：
{
  "date": "2026-04-14",
  "generated_at": "2026-04-14 21:35:36",
  "watch_list": [
    {
      "ticker": "AMZN",
      "model_score": 0.20720815658569336
    },
    {
      "ticker": "WFC",
      "model_score": 0.17481085658073425
    },
    {
      "ticker": "NVDA",
      "model_score": 0.15905532240867615
    },
    {
      "ticker": "MS",
      "model_score": 0.14602988958358765
    },
    {
      "ticker": "META",
      "model_score": 0.10776114463806152
    }
  ]
}


/opt/anaconda3/envs/mlenv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
